In [ ]:
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# LOAD — same header/data mismatch as every other InPlace export (7 header
# names, 8 fields per row due to a trailing per-student sequence counter).
# index_col=False keeps the real 7 columns aligned and drops the counter.
# ─────────────────────────────────────────────────────────────────────────────
apps = pd.read_csv('../data/CL26StudentOpportunityApplications (2).csv', index_col=False)
ph = pd.read_csv('../data/CareerLaunch2026PlacementHistory.csv', index_col=False)

print("=== Raw applications file ===")
print("Total rows:", len(apps))
print("Unique students:", apps['Student Code'].nunique())
print("Unique opportunities:", apps['Opportunity Id'].nunique())

# ─────────────────────────────────────────────────────────────────────────────
# SCOPE TO CL26 — placement history is ground truth for "real CL26 2026
# opportunity, any of the four hubs." Anything else (other programs/years,
# test entries) gets dropped by the inner join, regardless of how the
# applications export itself was filtered upstream in InPlace.
# ─────────────────────────────────────────────────────────────────────────────
cl26_opps = ph[['Opportunity Id', 'Group Name', 'Agency Id', 'Agency Name']].drop_duplicates('Opportunity Id')
scoped = apps.merge(cl26_opps, on='Opportunity Id', how='inner')

print(f"\nRows after scoping to CL26 ground-truth opportunities: {len(scoped)}")
print(f"Dropped as out-of-scope: {len(apps) - len(scoped)}")
print(f"Unique students after scoping: {scoped['Student Code'].nunique()}")

print("\n=== Scoped applications by hub ===")
print(scoped['Group Name'].value_counts(dropna=False))

print("\n=== Application status breakdown (scoped) ===")
print(scoped['Opportunity Application Status'].value_counts())

# ─────────────────────────────────────────────────────────────────────────────
# WHAT GOT DROPPED — sanity check that the scoping step is still doing real
# work even on an export that claims to already be filtered to 2026.
# ─────────────────────────────────────────────────────────────────────────────
opp_not_in_ph = set(apps['Opportunity Id']) - set(cl26_opps['Opportunity Id'])
orphans = apps[apps['Opportunity Id'].isin(opp_not_in_ph)]
print(f"\n=== Out-of-scope rows (Opportunity Id not in CL26 placement history): {len(orphans)} ===")
print(f"Unique students in those orphan rows: {orphans['Student Code'].nunique()}")
print(orphans['Opportunity Application Status'].value_counts())

# ─────────────────────────────────────────────────────────────────────────────
# CANONICAL FINAL PLACEMENTS — same one-student-one-placement pipeline used
# throughout, needed to check the applicant funnel below.
# ─────────────────────────────────────────────────────────────────────────────
ph['Changed Date'] = pd.to_datetime(ph['Changed Date'], format='%m/%d/%Y')
ph['Student Code'] = ph['Student Code'].astype('Int64')
matched = ph.dropna(subset=['Student Code']).copy()
matched_sorted = matched.sort_values(['Opportunity Id', 'Student Code', 'Changed Date', 'Changed Time'])
latest = matched_sorted.groupby(['Opportunity Id', 'Student Code']).tail(1).copy()
confirmed = latest[latest['Placement Status'] == 'Confirmed'].copy()
confirmed = confirmed.sort_values(['Student Code', 'Changed Date', 'Changed Time']).groupby('Student Code').tail(1).copy()
confirmed_students = set(confirmed['Student Code'].astype(int).unique())

applicant_students = set(scoped['Student Code'].unique())

print("\n=== Applicant funnel ===")
print(f"Students who applied (clean, CL26-scoped): {len(applicant_students)}")
print(f"Of those, confirmed placement:             {len(applicant_students & confirmed_students)}")
print(f"Of those, NOT confirmed:                   {len(applicant_students - confirmed_students)}")
print(f"Confirmed students missing an application record: {len(confirmed_students - applicant_students)}")